## Map - Refine 요약

- Map: 문서를 각각 요약
- Refine: 새로운 문서를 하나씩 추가로 읽으면서 기존 요약을 수정, 보완

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_teddynote import logging
logging.langsmith("Map-Refine")

LangSmith 추적을 시작합니다.
[프로젝트명]
Map-Refine


In [7]:
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf").load()

docs = docs[2:8]

In [8]:
from langchain_openai import ChatOpenAI

map_llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-4o-mini"
)


In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

#map prompt
map_prompt = ChatPromptTemplate.from_template("""
다음 문서의 내용을 요약하세요.

문서:
{context}
""")

#map chain 생성
map_chain = map_prompt|map_llm|StrOutputParser()

In [9]:
# 첫번째 문서 요약
summary = map_chain.invoke({
    "context":docs[0].page_content
})

In [ ]:
#Refine 단계
#기존 요약 + 새로운 문서를 보고 기존 요약을 수정하고 보완

refine_prompt = ChatPromptTemplate.from_template("""
기존 요약:
{existing_summary}

새로운 문서:
{context}

새로운 문서의 내용을 기존 요약에 반영하여
더 정확하고 완전한 요약을 작성하세요.

기존 요약에서 새로운 문서와 관련 없는 내용은
가능하면 유지하고, 새로운 정보가 있다면 추가하세요.

한국어로 답변하세요.
""")

refine_chain = refine_prompt|map_llm|StrOutputParser()

In [12]:
#Refine 반복
#두번째 문서부터 하나씩 추가하면서 기존 요약 계속 보완

for doc in docs[1:]:
    summary=refine_chain.invoke({
        "existing_summary": summary,
        "context":doc.page_content
    })

In [13]:
print(summary)

문서의 제목은 "인공지능 산업 동향 브리프"로, 인공지능(AI) 산업의 최근 동향과 발전 상황을 다루고 있습니다. 이 브리프는 AI 기술의 발전, 시장 성장, 주요 기업의 활동, 정책 및 규제 변화 등을 포함하여 AI 산업의 전반적인 현황을 요약하고 분석합니다.

최근 미국 바이든 대통령은 '안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령'에 서명하였으며, 이 행정명령은 AI의 안전과 보안 기준 마련, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력 등을 골자로 하고 있습니다. 특히, AI 시스템의 안전성과 신뢰성을 확인하기 위한 표준 및 모범사례를 확립하고, AI의 무책임한 사용으로 인한 차별과 편견을 방지하기 위한 조치를 확대하는 내용이 포함되어 있습니다. 또한, 의료 분야에서 책임 있는 AI 사용을 촉진하고, AI로 인한 근로자 피해를 완화하기 위한 원칙과 모범사례를 마련하는 등의 노력이 강조되고 있습니다.

AI 연구를 촉진하기 위해 국가AI연구자원(NAIRR)을 통해 기술과 인프라를 지원하고, 외국인 전문가들이 미국에서 공부하고 취업할 수 있도록 비자 기준과 인터뷰 절차를 현대화하는 방안도 포함되어 있습니다. 이러한 정책 변화는 AI의 응용 분야와 미래 전망에 대한 논의에도 중요한 영향을 미칠 것으로 예상됩니다.

또한, G7은 2023년 10월 30일 '히로시마 AI 프로세스'를 통해 AI 기업을 대상으로 하는 국제 행동강령에 합의하였습니다. 이 행동강령은 첨단 AI 시스템의 위험 식별과 완화를 위한 자발적인 채택을 권고하며, AI 수명주기 전반에 걸친 위험 평가와 완화, 투명성과 책임성 보장, 정보공유 및 이해관계자 간 협력, 보안 통제, 콘텐츠 인증 및 출처 확인 등의 조치를 요구합니다. G7은 이 행동강령을 통해 AI 시스템의 개발 과정에서 위험을 평가하고 완화하는 조치를 채택하며, AI의 성능과 한계를 공개하고 적절한 사용 영역을 알리는 방법으로 투명성을 강화할 계획입니다.

최근 